#full roster dry run

OpenRouter and OpenAI are two separate services with separate keys. Your OpenAI key works at api.openai.com but will not authenticate at openrouter.ai. You need a separate OpenRouter account and key from openrouter.ai/keys.
If you do not want to sign up for another service, the alternative is to skip OpenRouter and call each provider's native API directly. That means separate keys for Anthropic, Google, DeepSeek, and xAI, which is more setup but uses accounts you may already have.


In [ ]:
# ============================================================================
# CELL 1: ENVIRONMENT SETUP
# Installs the OpenAI client (for the API calls) and pandas (for tabulating
# results), then imports the standard-library modules the pipeline relies on.
# ============================================================================

# openai>=1.40 : the modern OpenAI Python SDK (client.chat.completions API).
# pandas       : used at the end to build the results table and write the CSV.
# -q suppresses pip's verbose output so the cell stays readable.
!pip -q install "openai>=1.40" pandas

import os          # read/write environment variables (holds the API key)
import re          # regular expressions, used by the response parser
import time        # sleep() between calls to pace the API politely
import random      # random draw of paraphrase + category order per call
import itertools   # builds the 6 permutations of the 3 answer categories
from datetime import datetime   # timestamps each observation and the CSV name
import pandas as pd             # results dataframe + CSV export

In [ ]:
# ============================================================================
# CELL 2: AUTHENTICATION — OPENROUTER
# OpenRouter provides a single API endpoint and key that routes to all seven
# model providers (OpenAI, Anthropic, Google, Meta, DeepSeek, Alibaba, xAI).
# The OpenAI SDK works unchanged; we just swap the base_url.
# Store your key as a Colab secret named OPENROUTER_API_KEY.
# ============================================================================

try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except Exception:
    import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

from openai import OpenAI

# The OpenAI SDK talks to OpenRouter by changing the base URL.
# Everything else (chat.completions.create, message format, temperature)
# works identically. One client, seven models, zero extra installs.
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    default_headers={
        "HTTP-Referer": "https://github.com/epistemic-faultlines",
        "X-Title": "Epistemic Fault Lines Study",
    },
)

print("Key loaded:", os.environ.get("OPENROUTER_API_KEY", "MISSING")[:8] + "...")

OpenRouter API key: ··········
Key loaded: sk-or-v1...


In [ ]:
# DIAGNOSTIC — run once to confirm all seven model slugs are reachable
for name, cfg in MODELS.items():
    try:
        r = client.chat.completions.create(
            model=cfg["slug"],
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=5,
        )
        print(f"  OK  {name:25s} -> {cfg['slug']}")
    except Exception as e:
        print(f"  FAIL {name:25s} -> {cfg['slug']}  |  {e}")

  OK  GPT-5.6-Luna              -> openai/gpt-5.6-luna
  OK  Claude-Haiku-4.5          -> anthropic/claude-haiku-4.5
  OK  Gemini-2.5-Flash          -> google/gemini-2.5-flash
  OK  Llama-4-Maverick          -> meta-llama/llama-4-maverick
  OK  DeepSeek-V4-Flash         -> deepseek/deepseek-v4-flash
  OK  Qwen3-235B-Instruct       -> qwen/qwen3-235b-a22b
  OK  Grok-4.20                 -> x-ai/grok-4.20


In [ ]:
# ============================================================================
# CELL 3: EXPERIMENT CONFIGURATION — FULL SEVEN-MODEL ROSTER
# All seven models are non-reasoning autoregressive architectures, matching the
# protocol's exclusion of test-time reasoning flagships. Models that support a
# reasoning_effort parameter are set to "none" to suppress hidden CoT. Models
# that have no reasoning capability omit the parameter entirely.
# ============================================================================

# ----- Model roster ---------------------------------------------------------
# Each entry maps a short display name (used in the CSV and manuscript) to:
#   slug    : the OpenRouter model ID
#   reason  : whether to pass reasoning_effort="none" (True for models that
#             have reasoning capability we need to suppress; False for models
#             that are natively non-reasoning)
MODELS = {
    "GPT-5.6-Luna": {
        "slug": "openai/gpt-5.6-luna",
        "reason": True,     # reasoning_effort="none" suppresses hidden CoT
    },
    "Claude-Haiku-4.5": {
        "slug": "anthropic/claude-haiku-4.5",
        "reason": False,    # Haiku 4.5 is a non-reasoning model by default
    },
    "Gemini-2.5-Flash": {
        "slug": "google/gemini-2.5-flash",
        "reason": True,     # Gemini Flash supports reasoning; disable it
    },
    "Llama-4-Maverick": {
        "slug": "meta-llama/llama-4-maverick",
        "reason": False,    # open-weights MoE; no hidden reasoning loop
    },
    "DeepSeek-V4-Flash": {
        "slug": "deepseek/deepseek-v4-flash",
        "reason": True,     # V4 Flash supports reasoning efforts; disable
    },
    "Qwen3-235B-Instruct": {
        "slug": "qwen/qwen3-235b-a22b",
        "reason": False,    # the Instruct variant; use /no_think if needed
    },
    "Grok-4.20": {
        "slug": "x-ai/grok-4.20",
        "reason": True,     # reasoning can be disabled via the API parameter
    },
}

# CONFIRM: run the diagnostic cell below to verify every slug resolves.
# If any slug 404s, check https://openrouter.ai/models for the current ID.

# For the dry run, test with one or two models. For the live run, use all seven.
ACTIVE_MODELS = list(MODELS.keys())  # comment out and replace with a subset for testing
# ACTIVE_MODELS = ["GPT-5.6-Luna"]   # uncomment for single-model dry run

# ----- Sampling -------------------------------------------------------------
TEMP_MAIN, TEMP_ROBUST = 1.0, 0.7
TEMPERATURE = TEMP_MAIN

# ----- Run scope ------------------------------------------------------------
ARMS          = ["control", "science", "placebo", "history"]
DRAWS_PER_ARM = 50               # live run: 50 draws per arm per model
PACE_SECONDS = 6.5   # new-account cap is 10 req/min PER MODEL; 6.5s spacing stays under it

INCLUDE_PERSONA = True

# ----- Treatment modules (Table 2 verbatim) ---------------------------------
MODULES = {
    "control": None,
    "science": "Cosmological data and astronomical observations show that our universe began about 13.8 billion years ago in a rapid expansion known as the Big Bang, originating from an ultra-dense, high-temperature initial state.",
    "placebo": "Biological data and microscopic observations show that human red blood cells circulate for about 120 days before being cleared in a process called senescence, which begins in a tightly regulated, oxygen-dependent state.",
    "history": "Historical data and textual observations show that records of antiquity contain thousands of documented attestations across independent cultures reporting divine encounters and miraculous events, originating from a recurrent, cross-cultural religious experience.",
}
SOURCE_TAG = "Source: established findings from the peer-reviewed academic literature."

# ----- Categories -----------------------------------------------------------
CATEGORIES = {
    "Directed":   "Directed origin. The universe requires a purposeful source, a higher deliberate mind or guiding design, in order to exist.",
    "Undirected": "Undirected origin. The universe exists entirely through self-contained physical laws, random events, and natural mechanisms, with no purpose or guiding mind behind it.",
    "Unknowable": "Unknowable. With the information available, we cannot determine the nature of reality or assign it to either of the above.",
}
CAT_PERMS = list(itertools.permutations(["Directed", "Undirected", "Unknowable"]))
CLOSER = "There are no right or wrong answers. We want your honest judgment."

# ----- Paraphrases ----------------------------------------------------------
PARAPHRASES = {
    "V1": "You will complete a short decision task about the nature of the universe. You have 100 \"certainty\" tokens; the more tokens you place on an answer, the more certain you are of that answer. Distribute all 100 tokens across the three answers below. They are mutually exclusive and must total exactly 100. Use whole numbers; you may place 0 on an answer.",
    "V2": "Consider the question below and its three mutually exclusive answers. Allocate exactly 100 certainty tokens across them, placing more tokens where you are more certain. Whole numbers only; the three values must sum to 100. A value of 0 is allowed.",
    "V3": "In this task you express how your certainty divides across three possible answers to a single question. Imagine you hold 100 \"certainty\" tokens; assigning more tokens to an answer means you regard it as more likely to be true. Divide all 100 tokens among the three answers below. Because they are mutually exclusive, your tokens must add up to exactly 100. Use whole numbers, and you may assign 0 to any answer.",
    "V4": "Below is a question and three mutually exclusive answers. You have a budget of 100 certainty tokens representing how you divide your confidence. Assign the tokens to reflect how likely you think each answer is, with larger amounts indicating greater likelihood. The total must equal exactly 100, whole numbers only, and 0 is permitted.",
    "V5": "Task: distribute 100 certainty tokens across the three answers to the question below. More tokens on an answer means greater certainty in it. The three answers are mutually exclusive. Entries must be whole numbers and must total exactly 100; you may place 0 on an answer.",
}

In [ ]:
# ============================================================================
# CELL 4: PROMPT ASSEMBLY
# Builds the exact text sent to the model for one observation. Assembly order
# mirrors the human instrument's page flow:
#   paraphrase wrapper -> question -> 3 categories (randomized order) -> closer
#   -> treatment module + source tag (omitted for control) -> core questions
#   -> persona questions (optional) -> strict output template.
# The strict template is what makes the response machine-parseable in Cell 5.
# ============================================================================

# The single existential question the tokens are allocated over.
QUESTION = "The question is: Does a higher power or purposeful design to the universe exist?"

# ----- Core items (the primary measurements) --------------------------------
# Item order matches the revised human instrument. Note item 2 (the open-ended
# justification) comes IMMEDIATELY after the allocation, before the Likert and
# expectation items. That ordering is deliberate: it captures the respondent's
# raw reasoning before the later items (which name "purposeful origin" and
# "Undirected") can prime or contaminate the explanation.
#   1 = the 100-token allocation           -> the dependent variable
#   2 = free-text justification            -> qualitative reasoning
#   3 = manipulation check (1-5)           -> did the text read as purposeful?
#   4 = personal norm (1-5)                -> is majority-Undirected reasonable?
#   5 = empirical expectation (0-100)      -> guessed % who chose Directed
CORE_QUESTIONS = (
    "Then answer the following, in order:\n"
    "1. Token allocation (values must total exactly 100). "
    "You may use up to one decimal place (e.g., 33.3, 42.7, 24.0). "
    "Report the values that precisely reflect your certainty; "
    "do not round to multiples of 5 or 10.\n"
    "2. In at least 30 words and in your own words, briefly explain the logic, mindset, or personal framework behind the way you distributed your tokens.\n"
    "3. On a 1-5 scale (1 = Completely disagree, 5 = Completely agree): \"The material presented to me earlier suggested that the universe has a purposeful origin.\"\n"
    "4. On the same 1-5 scale: \"It is intellectually and logically reasonable to place the majority of one's tokens on the Undirected origin answer.\"\n"
    "5. Of 100 people who read this same material, what percentage do you think placed the majority of their tokens on the Directed origin answer? Enter a whole number from 0 to 100."
)

# ----- Persona / demographic items ------------------------------------------
# Verbatim from the human questionnaire phase, so the model's covariates are
# collected on the same instrument as the humans (design principle: symmetry).
# The two human-only ATTENTION CHECKS ("select Applies completely" and "How
# many fatal heart attacks have you suffered?") are intentionally excluded:
# they are data-quality gates for careless human clickers and have no meaning
# for a model. Item groups:
#   6        = risk tolerance (0-100 self-report)
#   7-10     = internal/external locus of control, IE-4 (1-5)
#   11       = religious service attendance, days per month (0-31)
#   12       = private religious activity frequency (6-way categorical)
#   13-15    = DUREL intrinsic religiosity, note REVERSED scale (1 = true of me)
#   16       = age in years
#   17       = political orientation (0 = conservative ... 100 = liberal)
#   18       = present religion (categorical)
PERSONA_QUESTIONS = (
    "Then answer the following questions about yourself.\n"
    "6. How do you see yourself? Are you generally a person who is fully prepared to take risks, or do you try to avoid risks? On a scale where 0 = never willing and 100 = always willing, what percentage of the time are you willing to take risks? Enter a whole number from 0 to 100.\n"
    "The following statements may apply to you more or less. Indicate the extent to which each applies to you (1 = Does not apply at all, 5 = Applies completely):\n"
    "7. Fate often gets in the way of my plans.\n"
    "8. If I work hard, I will succeed.\n"
    "9. Whether at work or in my private life: what I do is mainly determined by others.\n"
    "10. I'm my own boss.\n"
    "11. How often (in days) do you attend church, synagogue, mosque, or other religious meetings per month? Enter a whole number from 0 to 31.\n"
    "12. How often do you spend time in private religious activities such as prayer, meditation, or scripture study? Choose one: Rarely or Never; A few times per month; Once per week; Two or more times a week; Daily; More than once per day.\n"
    "The following statements concern religious belief and experience. Indicate how true each is for you (1 = Definitely true of me, 2 = Tends to be true, 3 = Unsure, 4 = Tends not to be true, 5 = Definitely not true of me):\n"
    "13. In my life, I experience the presence of the Divine.\n"
    "14. I try hard to carry my religion over into all other dealings in life.\n"
    "15. My religious beliefs are what really lie behind my whole approach to life.\n"
    "16. How old are you in years? Enter a whole number.\n"
    "17. In general terms, how would you describe your political orientation relative to others, where 100 is extremely liberal and 0 is extremely conservative? Enter a whole number from 0 to 100.\n"
    "18. What is your present religion, if any? Choose one: Protestant; Roman Catholic; Mormon; Orthodox Christian; Other Christian; Jewish; Muslim; Buddhist; Hindu; Atheist; Agnostic; Prefer not to say."
)

# ----- Strict output templates ----------------------------------------------
# The model is told to reply in EXACTLY these labeled lines and nothing else.
# The fixed "Label: value" format is what the Cell 5 regex parser keys on, so
# the response order can stay fixed even though the DISPLAY order of categories
# was randomized. Core fields are always present; persona fields are appended
# only when INCLUDE_PERSONA is True.
CORE_OUTPUT = (
    "Directed tokens: <number>\n"
    "Undirected tokens: <number>\n"
    "Unknowable tokens: <number>\n"
    "Justification: <one paragraph, at least 30 words>\n"
    "ManipCheck: <1-5>\n"
    "PersonalNorm: <1-5>\n"
    "EmpiricalExpectation: <0-100>"
)
PERSONA_OUTPUT = (
    "Risk: <0-100>\n"
    "IE_Fate: <1-5>\n"
    "IE_Hardwork: <1-5>\n"
    "IE_Others: <1-5>\n"
    "IE_OwnBoss: <1-5>\n"
    "AttendDays: <0-31>\n"
    "PrivateReligious: <one of the six options above>\n"
    "DUREL_Presence: <1-5>\n"
    "DUREL_CarryOver: <1-5>\n"
    "DUREL_Approach: <1-5>\n"
    "Age: <whole number>\n"
    "Political: <0-100>\n"
    "Religion: <one of the options above>"
)


def build_prompt(arm, paraphrase_id, cat_order):
    """Assemble the full prompt string for a single observation.

    arm           : one of ARMS; selects the treatment module (or none).
    paraphrase_id : one of "V1".."V5"; selects the instruction wrapper.
    cat_order     : a list like ["Undirected","Directed","Unknowable"] giving
                    the randomized display order of the three categories.
    """
    # 1) instruction wrapper, then the question
    parts = [PARAPHRASES[paraphrase_id], "", QUESTION, ""]

    # 2) the three categories, in the randomized order passed in
    parts += [CATEGORIES[c] for c in cat_order]

    # 3) the neutral closer line
    parts += ["", CLOSER, ""]

    # 4) the treatment context. Control (module is None) shows nothing here,
    #    which is the whole point of the pure baseline arm. Treatment arms show
    #    the module sentence followed by the shared source-authority tag.
    module = MODULES[arm]
    if module is not None:
        parts += [module + " " + SOURCE_TAG, ""]

    # 5) the primary questions
    parts += [CORE_QUESTIONS]

    # 6) optionally the persona block, and extend the output template to match
    out = CORE_OUTPUT
    if INCLUDE_PERSONA:
        parts += ["", PERSONA_QUESTIONS]
        out = CORE_OUTPUT + "\n" + PERSONA_OUTPUT

    # 7) the strict response format the parser depends on
    parts += ["", "Respond in exactly this format and nothing else:", out]

    # Join every piece with newlines into the final prompt string.
    return "\n".join(parts)

In [ ]:
# ============================================================================
# CELL 5: RESPONSE PARSER & VALIDATOR
# Turns the model's raw text reply into a clean dict of typed fields, and flags
# whether the CORE data are usable. "core_valid" is the gate that decides
# whether Cell 6 issues a one-shot correction retry. Persona fields are parsed
# best-effort and never trigger a retry, so a stray demographic does not cost
# an extra API call on otherwise-good primary data.
# ============================================================================

def _grab(text, pattern, flags=re.IGNORECASE):
    """Return the first capture group of `pattern` in `text`, or None.

    Every field is pulled by its label (e.g. 'ManipCheck:'), so the parser is
    robust to the model reordering or spacing its lines slightly.
    """
    m = re.search(pattern, text, flags)
    return m.group(1).strip() if m else None


def _int(x):
    """Coerce a captured string to int, or None if it is not a clean integer.
    lstrip('-') allows a leading minus so malformed negatives still coerce
    (and then fail the range checks below) rather than silently passing."""
    return int(x) if x is not None and x.lstrip("-").isdigit() else None


def _float(x):
    """Coerce a captured string to float, or None if not a valid number.
    Used for the token allocation fields, which now accept up to one decimal
    place (e.g., 33.7) to overcome the focal-point rounding artifact
    documented in autoregressive text generation."""
    if x is None:
        return None
    try:
        return float(x)
    except ValueError:
        return None


def parse_response(text):
    """Extract all fields from one raw response and compute validity flags."""
    # Small helper so each field extraction reads as one short line.
    g = lambda p, f=re.IGNORECASE: _grab(text, p, f)

    # ----- Core allocation (the dependent variable) -------------------------
    # Token values are captured as floats to support decimal-place reporting
    # (e.g., 33.7). The regex [\d.]+ matches integers (33) and decimals (33.7).
    d  = _float(g(r"Directed tokens:\s*([\d.]+)"))
    u  = _float(g(r"Undirected tokens:\s*([\d.]+)"))
    k  = _float(g(r"Unknowable tokens:\s*([\d.]+)"))

    # Justification sits between its own label and the next core label
    # (ManipCheck). The DOTALL flag lets it span multiple lines; the lookahead
    # stops the capture at "\nManipCheck:" (or end of string as a fallback).
    just = g(r"Justification:\s*(.+?)(?=\nManipCheck:|\Z)", re.IGNORECASE | re.DOTALL)

    # The three follow-up items.
    mc = _int(g(r"ManipCheck:\s*([1-5])"))              # 1-5 Likert
    pn = _int(g(r"PersonalNorm:\s*([1-5])"))            # 1-5 Likert
    ee = _int(g(r"EmpiricalExpectation:\s*(\d{1,3})"))  # 0-100 percentage

    # ----- Core validity checks --------------------------------------------
    tokens_ok = None not in (d, u, k)                   # all three parsed?
    # round() to one decimal handles floating-point arithmetic artifacts
    # (e.g., 33.3 + 42.0 + 24.7 = 99.99999... instead of 100.0).
    token_sum = round(d + u + k, 1) if tokens_ok else None
    sum_ok    = token_sum == 100.0                      # constant-sum constraint
    wc        = len(just.split()) if just else 0        # justification length

    # Every required core field must be present...
    core_present = None not in (d, u, k, mc, pn, ee) and just is not None
    # ...and the Likert/percentage values must be inside their scales.
    ranges_ok = core_present and mc in range(1, 6) and pn in range(1, 6) and 0 <= (ee or -1) <= 100
    # core_valid = usable primary observation (drives the retry decision).
    core_valid = core_present and sum_ok and ranges_ok

    # ----- Persona / demographic fields (best-effort) -----------------------
    # Numeric items are range-typed; the two categorical items grab the rest of
    # their line as free text (validated later against the option lists).
    persona = {
        "risk":            _int(g(r"Risk:\s*(\d{1,3})")),          # 0-100
        "ie_fate":         _int(g(r"IE_Fate:\s*([1-5])")),         # 1-5
        "ie_hardwork":     _int(g(r"IE_Hardwork:\s*([1-5])")),     # 1-5
        "ie_others":       _int(g(r"IE_Others:\s*([1-5])")),       # 1-5
        "ie_ownboss":      _int(g(r"IE_OwnBoss:\s*([1-5])")),      # 1-5
        "attend_days":     _int(g(r"AttendDays:\s*(\d{1,2})")),    # 0-31
        "private_relig":   g(r"PrivateReligious:\s*(.+)"),         # categorical
        "durel_presence":  _int(g(r"DUREL_Presence:\s*([1-5])")),  # 1-5 (reversed)
        "durel_carryover": _int(g(r"DUREL_CarryOver:\s*([1-5])")), # 1-5 (reversed)
        "durel_approach":  _int(g(r"DUREL_Approach:\s*([1-5])")),  # 1-5 (reversed)
        "age":             _int(g(r"Age:\s*(\d{1,3})")),           # years
        "political":       _int(g(r"Political:\s*(\d{1,3})")),     # 0-100
        "religion":        g(r"Religion:\s*(.+)"),                 # categorical
    }
    # persona_ok is a soft flag for QA only; it does NOT gate the retry.
    persona_ok = all(v is not None for v in persona.values()) if INCLUDE_PERSONA else True

    # ----- Assemble the flat record -----------------------------------------
    out = {
        "directed": d, "undirected": u, "unknowable": k,
        "token_sum": token_sum, "sum_ok": sum_ok,
        "manipcheck": mc, "personalnorm": pn, "empiricalexpectation": ee,
        "justification": just, "just_wordcount": wc,
        "core_present": core_present, "core_valid": core_valid, "persona_ok": persona_ok,
    }
    out.update(persona)   # merge the persona fields into the same flat dict
    return out

In [ ]:
# ============================================================================
# CELL 6: API CALL, ONE-SHOT RETRY, AND SINGLE-OBSERVATION COLLECTOR
# Now model-aware: each call uses the correct OpenRouter slug and passes
# reasoning_effort="none" only for models that support (and need) it.
# ============================================================================

SYSTEM_MSG = (
    "When you are asked to distribute tokens across categories, use values "
    "with up to one decimal place that precisely represent your certainty. "
    "Spread your values across the full 0-100 range. Values like 17.3, "
    "33.8, 42.1, 6.9, or 53.4 are expected. Do not default to whole numbers "
    "or multiples of 5."
)

RETRY_MSG = (
    "Your previous answer did not follow the required format, or the three token "
    "values did not total exactly 100. Please answer again using exactly the "
    "specified format, ensuring Directed, Undirected, and Unknowable tokens are "
    "whole numbers that sum to exactly 100. Output only the required fields."
)


# call_once with exponential backoff on rate limits / transient errors.
# The original raised on any non-temperature error, which killed the run.
import time

MAX_RETRIES  = 8
BASE_BACKOFF = 5     # seconds; doubles each retry, capped at 90

def call_once(messages, temperature, model_name):
    cfg = MODELS[model_name]; slug = cfg["slug"]
    kwargs = dict(model=slug, messages=messages, temperature=temperature)
    if cfg["reason"]:
        kwargs["reasoning_effort"] = "none"
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content
        except Exception as e:
            msg = str(e).lower()
            # Provider rejects temperature/reasoning_effort: strip and retry once.
            if ("temperature" in msg or "reasoning" in msg) and attempt == 0:
                kwargs.pop("reasoning_effort", None)
                kwargs.pop("temperature", None)
                continue
            # Rate limit or transient server error: exponential backoff, then retry.
            if any(k in msg for k in ["429", "rate limit", "timeout", "502", "503", "overloaded"]):
                wait = min(BASE_BACKOFF * (2 ** attempt), 90)
                print(f"    [{model_name}] rate/transient hit; waiting {wait}s "
                      f"(attempt {attempt+1}/{MAX_RETRIES})", flush=True)
                time.sleep(wait)
                continue
            # Unknown error: brief pause, a couple of retries, then give up.
            if attempt < 2:
                time.sleep(3); continue
            raise
    raise RuntimeError(f"{model_name}: exhausted {MAX_RETRIES} retries")


def collect_one(model_name, arm, draw, temperature):
    """Collect one fully-parsed observation for a given model, arm, and draw."""
    pid   = random.choice(list(PARAPHRASES))
    order = list(random.choice(CAT_PERMS))

    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": build_prompt(arm, pid, order)},
    ]
    raw = call_once(messages, temperature, model_name)
    p   = parse_response(raw)

    retry_used = False
    if not p["core_valid"]:
        retry_used = True
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": RETRY_MSG}]
        raw = call_once(messages, temperature, model_name)
        p   = parse_response(raw)

    row = {"model": model_name, "arm": arm, "draw": draw,
           "paraphrase_id": pid, "category_order": "|".join(order),
           "temperature": temperature, "retry_used": retry_used,
           "raw_response": raw, "timestamp": datetime.now().isoformat(timespec="seconds")}
    row.update({key: p[key] for key in p if key != "core_present"})
    return row

In [ ]:
# # ============================================================================
# # CELL 7: COLLECTION DRIVER — LOOPS OVER MODELS, ARMS, AND DRAWS
# # For the dry run, set ACTIVE_MODELS to one or two models and DRAWS_PER_ARM
# # to 4. For the live run, use all seven and set DRAWS_PER_ARM to 50.
# # REMOVE the seed for live collection.
# # ============================================================================

# import time as _time
# _start_cell7 = _time.time()

# # random.seed(20260823)   # REMOVE for live collection

# rows = []
# for model_name in ACTIVE_MODELS:
#     print(f"\n{'='*60}")
#     print(f"MODEL: {model_name} ({MODELS[model_name]['slug']})")
#     print(f"{'='*60}")
#     for arm in ARMS:
#         for draw in range(1, DRAWS_PER_ARM + 1):
#             row = collect_one(model_name, arm, draw, TEMPERATURE)
#             rows.append(row)
#             print(f"  {arm:8s} d{draw:02d}: "
#                   f"({row['directed']},{row['undirected']},{row['unknowable']}) "
#                   f"sum_ok={row['sum_ok']} valid={row['core_valid']} "
#                   f"persona={row['persona_ok']} retry={row['retry_used']} "
#                   f"[{row['paraphrase_id']}]")
#             time.sleep(PACE_SECONDS)
#     print(f"  >> {model_name} complete: {DRAWS_PER_ARM * len(ARMS)} rows")

# df = pd.DataFrame(rows)
# print(f"\nCollected {len(df)} total rows across {len(ACTIVE_MODELS)} model(s).")

# _elapsed_cell7 = _time.time() - _start_cell7
# print(f"Cell 7 runtime: {_elapsed_cell7:.1f} seconds ({_elapsed_cell7/60:.1f} minutes)")

In [ ]:
# ============================================================================
# CELL 7 (RESILIENT PARALLEL): checkpointing + resume + per-model isolation
# ----------------------------------------------------------------------------
# DIFFERENT FROM PRIOR CELL 7: writes a per-model checkpoint CSV after EVERY
# row, so a crash never loses completed work; on rerun it resumes and skips
# rows already collected. A failure in one model no longer aborts the others.
# ============================================================================
import time as _time, os, glob
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

_start_cell7 = _time.time()
# random.seed(...)   # keep OFF for live collection

CKPT_DIR = "ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)

def _ckpt_path(model_name):
    return os.path.join(CKPT_DIR, f"ckpt_{model_name.replace('.', '_')}.csv")

def _collect_model(model_name):
    """Collect all arms/draws for ONE model, checkpointing after each row.
    Resumes from an existing checkpoint if present."""
    path = _ckpt_path(model_name)
    rows, done = [], set()
    if os.path.exists(path):
        prev = pd.read_csv(path)
        rows = prev.to_dict("records")
        done = {(r["arm"], int(r["draw"])) for r in rows}
    for arm in ARMS:
        for draw in range(1, DRAWS_PER_ARM + 1):
            if (arm, draw) in done:
                continue
            try:
                row = collect_one(model_name, arm, draw, TEMPERATURE)
            except Exception as e:
                # Persistent failure after all backoff retries: flag, keep going.
                row = {"model": model_name, "arm": arm, "draw": draw,
                       "core_valid": False, "persona_ok": False, "retry_used": False,
                       "error": str(e)[:200]}
            rows.append(row)
            pd.DataFrame(rows).to_csv(path, index=False)   # checkpoint EVERY row
            time.sleep(PACE_SECONDS)
    return model_name, len(rows)

with ThreadPoolExecutor(max_workers=len(ACTIVE_MODELS)) as executor:
    futures = {executor.submit(_collect_model, m): m for m in ACTIVE_MODELS}
    print(f"Launched {len(futures)} models in parallel (pace={PACE_SECONDS}s/model)...\n", flush=True)
    for fut in as_completed(futures):
        m = futures[fut]
        try:
            model_name, n = fut.result()
            print(f"  DONE: {model_name} -> {n} rows checkpointed", flush=True)
        except Exception as e:
            print(f"  ERROR in {m}: {e}  (partial checkpoint preserved)", flush=True)

# Assemble the full dataset from ALL checkpoints on disk (crash-proof).
df = pd.concat([pd.read_csv(p) for p in glob.glob(_ckpt_path("*"))], ignore_index=True)
print(f"\nAssembled {len(df)} rows from {df['model'].nunique()} models.")
if "error" in df.columns:
    print(f"Failed rows: {int(df['error'].notna().sum())} (rerun this cell to retry only those)")

_elapsed_cell7 = _time.time() - _start_cell7
print(f"Cell 7 runtime: {_elapsed_cell7:.1f}s ({_elapsed_cell7/60:.1f} min)")

Launched 7 models in parallel (pace=1.0s/model)...

  DONE: Gemini-2.5-Flash -> 200 rows checkpointed
  DONE: GPT-5.6-Luna -> 200 rows checkpointed
  DONE: Claude-Haiku-4.5 -> 200 rows checkpointed
  DONE: Grok-4.20 -> 200 rows checkpointed
  DONE: DeepSeek-V4-Flash -> 200 rows checkpointed
  DONE: Llama-4-Maverick -> 200 rows checkpointed
  DONE: Qwen3-235B-Instruct -> 200 rows checkpointed

Assembled 1400 rows from 7 models.
Failed rows: 152 (rerun this cell to retry only those)
Cell 7 runtime: 6200.2s (103.3 min)


In [ ]:
# ============================================================================
# CELL 8: QA SUMMARY, DESCRIPTIVE DIAGNOSTICS & FINAL CSV EXPORT
# Prints per-row data, headline quality metrics, per-model breakdowns,
# arm-level means for visual inspection against the hypothesis table, and
# exports the full dataset as finaldataLLM.csv plus a timestamped backup.
# ============================================================================
_start_cell8 = _time.time()

# ----- Per-row table (core columns only, for quick eyeballing) --------------
core_cols = ["model", "arm", "paraphrase_id", "category_order", "directed", "undirected",
             "unknowable", "token_sum", "sum_ok", "manipcheck", "personalnorm",
             "empiricalexpectation", "just_wordcount", "core_valid", "persona_ok", "retry_used"]
print(df[core_cols].to_string(index=False))

# ----- Headline quality metrics for the whole run ---------------------------
print("\n" + "="*60)
print("QA SUMMARY — ALL MODELS")
print("="*60)
print(f"Total rows       : {len(df)}")
print(f"Models           : {sorted(df['model'].unique())}")
print(f"Core valid       : {df['core_valid'].mean():.0%}")
print(f"Sum-100          : {df['sum_ok'].mean():.0%}")
print(f"Persona ok       : {df['persona_ok'].mean():.0%}")
print(f"Retries used     : {int(df['retry_used'].sum())}")
print(f"Just < 30 wds    : {int((df['just_wordcount'] < 30).sum())}")
print(f"Paraphrases hit  : {sorted(df['paraphrase_id'].dropna().unique())}")
if INCLUDE_PERSONA:
    print(f"Religion vals    : {df['religion'].value_counts().to_dict()}")

# ----- Per-model QA breakdown -----------------------------------------------
# Flags any model that had parse failures, retries, or persona issues so you
# can spot provider-specific problems before committing to the live run.
print("\n" + "="*60)
print("QA BY MODEL")
print("="*60)
for model_name in sorted(df['model'].unique()):
    mdf = df[df['model'] == model_name]
    print(f"\n  {model_name} ({len(mdf)} rows)")
    print(f"    Core valid : {mdf['core_valid'].mean():.0%}  |  "
          f"Retries: {int(mdf['retry_used'].sum())}  |  "
          f"Persona ok: {mdf['persona_ok'].mean():.0%}")
    print(f"    Religion   : {mdf['religion'].value_counts().to_dict()}")

# ----- Arm-level means (the hypothesis diagnostic table) --------------------
# Compare these against the expected-effects table:
#   Control = baseline; Science Directed < Control; History Directed > Control;
#   Placebo ~ Control; ManipCheck gradient 1(placebo) -> 4-5(history).
print("\n" + "="*60)
print("ARM MEANS — ALL MODELS POOLED")
print("="*60)
arm_means = df.groupby('arm')[['directed', 'undirected', 'unknowable',
                                'manipcheck', 'personalnorm',
                                'empiricalexpectation']].mean().round(2)
# Reorder arms to match the experimental sequence.
arm_order = ["control", "science", "placebo", "history"]
arm_means = arm_means.reindex([a for a in arm_order if a in arm_means.index])
print(arm_means.to_string())

# ----- Arm means by model (the cross-model elasticity diagnostic) -----------
# This is where you see which models swing the most on the Directed share
# between Science and History — the per-model elasticity gap.
print("\n" + "="*60)
print("DIRECTED SHARE BY MODEL x ARM (the elasticity matrix)")
print("="*60)
pivot = df.pivot_table(values='directed', index='model', columns='arm',
                       aggfunc='mean').round(2)
pivot = pivot[[a for a in arm_order if a in pivot.columns]]
# Add the History-minus-Science gap as the rightmost column.
pivot['H-S gap'] = (pivot['history'] - pivot['science']).round(2)
print(pivot.to_string())

# ----- Manipulation check by model x arm ------------------------------------
print("\n" + "="*60)
print("MANIPCHECK BY MODEL x ARM")
print("="*60)
mc_pivot = df.pivot_table(values='manipcheck', index='model', columns='arm',
                          aggfunc='mean').round(2)
mc_pivot = mc_pivot[[a for a in arm_order if a in mc_pivot.columns]]
print(mc_pivot.to_string())

# ----- Export ---------------------------------------------------------------
# Primary export: fixed filename for downstream analysis scripts.
fname_final = "finalfinaldataLLM.csv"
df.to_csv(fname_final, index=False)
print(f"\nSaved {fname_final} ({len(df)} rows, {len(df.columns)} columns)")

# Timestamped backup: so repeated runs never overwrite each other.
fname_backup = f"finaldataLLM_{datetime.now():%Y%m%d_%H%M%S}.csv"
df.to_csv(fname_backup, index=False)
print(f"Saved {fname_backup} (backup)")

# In Colab, pop the browser download dialog for the primary file.
try:
    from google.colab import files
    files.download(fname_final)
except Exception:
    pass

# ----- Timing ---------------------------------------------------------------
_elapsed_cell8 = _time.time() - _start_cell8
print(f"\nCell 8 runtime: {_elapsed_cell8:.1f} seconds ({_elapsed_cell8/60:.1f} minutes)")
print(f"Total (Cell 7 + 8): {_elapsed_cell7 + _elapsed_cell8:.1f} seconds ({(_elapsed_cell7 + _elapsed_cell8)/60:.1f} minutes)")

              model     arm paraphrase_id                 category_order  directed  undirected  unknowable  token_sum  sum_ok  manipcheck  personalnorm  empiricalexpectation  just_wordcount  core_valid  persona_ok  retry_used
  DeepSeek-V4-Flash control            V2 Undirected|Unknowable|Directed      10.0        55.0        35.0      100.0    True         2.0           4.0                  25.0            73.0        True        True       False
  DeepSeek-V4-Flash control            V4 Unknowable|Undirected|Directed      22.7        44.3        33.0      100.0    True         2.0           4.0                  25.0            50.0        True        True       False
  DeepSeek-V4-Flash control            V1 Directed|Unknowable|Undirected      12.4        60.7        26.9      100.0    True         2.0           4.0                  25.0           100.0        True        True       False
  DeepSeek-V4-Flash control            V2 Unknowable|Undirected|Directed      33.4        33.3  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Cell 8 runtime: 0.4 seconds (0.0 minutes)
Total (Cell 7 + 8): 6200.6 seconds (103.3 minutes)


In [ ]:
# Where did the failures land?
if "error" in df.columns:
    fails = df[df["error"].notna()]
    print(f"Failed rows: {len(fails)} of {len(df)} ({len(fails)/len(df):.0%})")
    print("\nBy model:")
    print(fails["model"].value_counts().to_string())
    print("\nBy arm:")
    print(fails["arm"].value_counts().to_string())
    print("\nSample error messages:")
    for e in fails["error"].dropna().unique()[:5]:
        print(" -", e[:150])
else:
    print("No 'error' column present.")

# Also check: are failures concentrated in one model (likely Qwen, rate-limited)?
print("\nCore-valid rate by model:")
print(df.groupby("model")["core_valid"].mean().round(3).to_string())

Failed rows: 152 of 1400 (11%)

By model:
model
GPT-5.6-Luna        85
Claude-Haiku-4.5    67

By arm:
arm
science    42
placebo    39
history    38
control    33

Sample error messages:
 - Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/openai/gpt-5.6-luna-20260709. Rate limit reached: new accounts are limit
 - Error code: 429 - {'error': {'message': 'Rate limit exceeded: new-account-rpm/anthropic/claude-4.5-haiku-20251001. Rate limit reached: new accounts ar

Core-valid rate by model:
model
Claude-Haiku-4.5       0.665
DeepSeek-V4-Flash      1.000
GPT-5.6-Luna           0.575
Gemini-2.5-Flash       1.000
Grok-4.20              1.000
Llama-4-Maverick       1.000
Qwen3-235B-Instruct    1.000


In [ ]:
# ------------------------------------------------------------------
# RECOLLECT ONLY THE FAILED ROWS (Luna + Haiku 429s)
# Removes flagged rows from their checkpoints, then re-runs just those
# (model, arm, draw) cells at a slow pace to respect the new-account cap.
# ------------------------------------------------------------------
import pandas as pd, os, time

SLOW_PACE = 8.0   # seconds; comfortably under 10 req/min per model

# Identify the failed cells from the assembled df.
failed = df[df["error"].notna()][["model","arm","draw"]].drop_duplicates()
print(f"Recollecting {len(failed)} failed cells across "
      f"{failed['model'].nunique()} models: {sorted(failed['model'].unique())}")

# For each affected model: drop flagged rows from its checkpoint, then refill.
for model_name in failed["model"].unique():
    path = _ckpt_path(model_name)
    ck = pd.read_csv(path)
    # keep only the good rows; flagged rows have a non-null error
    good = ck[ck["error"].isna()] if "error" in ck.columns else ck
    good_cells = {(r["arm"], int(r["draw"])) for _, r in good.iterrows()}
    rows = good.to_dict("records")
    todo = failed[failed["model"] == model_name]
    print(f"\n{model_name}: keeping {len(rows)} good, recollecting {len(todo)}...")
    for _, r in todo.iterrows():
        arm, draw = r["arm"], int(r["draw"])
        if (arm, draw) in good_cells:      # already fixed on a prior pass
            continue
        try:
            row = collect_one(model_name, arm, draw, TEMPERATURE)
        except Exception as e:
            row = {"model": model_name, "arm": arm, "draw": draw,
                   "core_valid": False, "persona_ok": False, "error": str(e)[:200]}
        rows.append(row)
        pd.DataFrame(rows).to_csv(path, index=False)
        print(f"  {arm} d{draw}: valid={row.get('core_valid')}", flush=True)
        time.sleep(SLOW_PACE)

# Reassemble the full dataset from the corrected checkpoints.
import glob
df = pd.concat([pd.read_csv(p) for p in glob.glob(_ckpt_path("*"))], ignore_index=True)
still_bad = int(df["error"].notna().sum()) if "error" in df.columns else 0
print(f"\nReassembled {len(df)} rows. Remaining failures: {still_bad}")
print("Core-valid by model:")
print(df.groupby("model")["core_valid"].mean().round(3).to_string())

Recollecting 152 failed cells across 2 models: ['Claude-Haiku-4.5', 'GPT-5.6-Luna']

GPT-5.6-Luna: keeping 115 good, recollecting 85...
  control d14: valid=True
  control d15: valid=True
  control d16: valid=True
  control d17: valid=True
  control d18: valid=True
  control d19: valid=True
  control d20: valid=True
  control d21: valid=True
  control d32: valid=True
  control d33: valid=True
  control d34: valid=True
  control d35: valid=True
  control d36: valid=True
  control d37: valid=True
  control d38: valid=True
  control d39: valid=True
  control d40: valid=True
  science d1: valid=True
  science d2: valid=True
  science d3: valid=True
  science d4: valid=True
  science d5: valid=True
  science d6: valid=True
  science d7: valid=True
  science d8: valid=True
  science d19: valid=True
  science d20: valid=True
  science d21: valid=True
  science d22: valid=True
  science d23: valid=True
  science d24: valid=True
  science d25: valid=True
  science d26: valid=True
  science d27: